# Step 1: lock in the RAMP baseline (official code), resumable via Hugging Face

**One-time setup**
1. Create a Hugging Face *write* token at https://huggingface.co/settings/tokens.
2. In Kaggle: *Add-ons → Secrets → Add secret*, name it `HF_TOKEN`, and enable it for this notebook.
3. Set `HF_REPO` below to `<your-hf-username>/aat-checkpoints`. It is created **private** on first push.

**Each session:** Accelerator *GPU T4 x2*, Internet *on*, then **Save Version → Save & Run All**. The run continues in the background. Each run stops cleanly after 11 h and pushes its checkpoint. Run the notebook again and it resumes from the Hub. Runs that are already finished are skipped automatically.

In [1]:
import os
os.environ['HF_REPO'] = 'matokebryan/aat-checkpoints'   # <- edit once
os.environ['TIME_BUDGET_H'] = '11'
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
%cd /kaggle/working
!git clone -q -b matt/funny-planck-3j7md5 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT
!pip install -q pyyaml
!bash scripts/ramp_official.sh setup

/kaggle/working
/kaggle/working/AAT
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
ERROR: Cannot install autoattack 0.1 (from git+https://github.com/fra31/auto-attack) and robustbench==1.1 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


## Sanity check: the official pretrained ℓ∞ model (expect ≈ 83.7 / 48.1 / 59.8 / 7.7 / 38.5)

In [2]:
if not os.path.exists('runs_official/pretr_linf/eval_autoattack.json'):
    !bash scripts/ramp_official.sh pretr 0

100%|█████████████████████████████████████████| 170M/170M [10:05<00:00, 282kB/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/AAT/aat/evaluate.py", line 123, in <module>
    main()
  File "/kaggle/working/AAT/aat/evaluate.py", line 118, in main
    evaluate(r, a.backend, a.attacks.split(","), a.n, a.bs, not a.no_unseen,
  File "/kaggle/working/AAT/aat/evaluate.py", line 73, in evaluate
    m = attack_batchwise(model, x, y, p, eps, backend, attacks, bs, dev, pgd_steps) & clean
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/AAT/aat/evaluate.py", line 30, in attack_batchwise
    from autoattack import AutoAttack
ModuleNotFoundError: No module named 'autoattack'


## A. Thesis baseline: RAMP from scratch (λ=5, 80 epochs, GP), the setting of thesis Table 7.1 / paper Table 3
Two seeds run in parallel, one per GPU. Re-run the notebook each session until both print `done already`. Then change the seeds to `"2" "3"`, then `"4"`.

In [3]:
!(bash scripts/ramp_official.sh train ramp_scratch "0" 0 & bash scripts/ramp_official.sh train ramp_scratch "1" 1 & wait)
!grep -h "epoch\]" external/ramp/trained_models/ramp_scratch_l5_s*/log_train.txt | tail -n 4   # epoch time -> sessions needed

>> ramp_scratch_l5_s0 on GPU 0
>> ramp_scratch_l5_s1 on GPU 1
grep: external/ramp/trained_models/ramp_scratch_l5_s*/log_train.txt: No such file or directory


In [4]:
# once the seeds are finished
!bash scripts/ramp_official.sh eval ramp_scratch "0 1" 0
!python scripts/compare_targets.py --runs runs_official --table thesis_table7_1 --map ramp_scratch_official=ramp
!python scripts/compare_targets.py --runs runs_official --table scratch_table3 --map ramp_scratch_official=ramp_l5

[hub] nothing to pull for ramp_official/ramp_scratch_l5_s0/ep_80_0.pth (RemoteEntryNotFoundError)
| run | n | metric | ours | paper | Δ | verdict |
|---|---|---|---|---|---|---|
| ramp_scratch_official | 0 | – | not evaluated | | | |

overall: OK
| run | n | metric | ours | paper | Δ | verdict |
|---|---|---|---|---|---|---|
| ramp_scratch_official | 0 | – | not evaluated | | | |

overall: OK


## B. Cheap cross-check: RAMP fine-tuning (paper Table 24), 5 seeds, about 1 GPU-hour each. Run it on the GPU time left over in a session

In [5]:
!(bash scripts/ramp_official.sh train ramp "0 2 4" 0 & bash scripts/ramp_official.sh train ramp "1 3" 1 & wait)
!(bash scripts/ramp_official.sh eval ramp "0 2 4" 0 & bash scripts/ramp_official.sh eval ramp "1 3" 1 & wait)
!python scripts/compare_targets.py --runs runs_official --map ramp_official=ramp_l1.5 pretr_linf=pretr_linf

>> ramp_ft_s1 on GPU 1
>> ramp_ft_s0 on GPU 0
[hub] nothing to pull for ramp_official/ramp_ft_s0/ep_3_0.pth (RemoteEntryNotFoundError)
[hub] nothing to pull for ramp_official/ramp_ft_s1/ep_3_0.pth (RemoteEntryNotFoundError)
| run | n | metric | ours | paper | Δ | verdict |
|---|---|---|---|---|---|---|
| ramp_official | 0 | – | not evaluated | | | |
| pretr_linf | 0 | – | not evaluated | | | |

overall: OK
